## Merge events

In [1]:
def build_timeline(events: list[dict]) -> list[dict]:
    if not events:
        return []
    results = []
    seen_timestamps = set()
    accepted_key = {"timestamp", "service", "message"}
    for event in events:
        if not accepted_key.issubset(event.keys()):
            raise ValueError(f"The event {event} does not have the complte keys")
        
        timestamp = event["timestamp"]
        service = event["service"]
        message = event["message"]

        if timestamp in seen_timestamps:
            for result in results:
                if result["timestamp"] != timestamp:
                    continue
                result["events"].append(
                    {
                        "service": service,
                        "message": message
                    }
                )
            continue
            
        seen_timestamps.add(timestamp)
        results.append(
            {
                "timestamp": timestamp,
                "events": [
                    {
                        "service": service,
                        "message": message
                    }
                ]
            }
        )

    # sorted by timestamp
    return sorted(results, key=lambda x: x["timestamp"])


In [2]:
# 1. Liste vide
assert build_timeline([]) == []


# 2. Un seul événement
assert build_timeline([
    {"timestamp": 10, "service": "api", "message": "ok"}
]) == [
    {
        "timestamp": 10,
        "events": [
            {"service": "api", "message": "ok"}
        ]
    }
]


# 3. Événements non triés
assert build_timeline([
    {"timestamp": 3, "service": "c", "message": "third"},
    {"timestamp": 1, "service": "a", "message": "first"},
    {"timestamp": 2, "service": "b", "message": "second"},
]) == [
    {
        "timestamp": 1,
        "events": [{"service": "a", "message": "first"}],
    },
    {
        "timestamp": 2,
        "events": [{"service": "b", "message": "second"}],
    },
    {
        "timestamp": 3,
        "events": [{"service": "c", "message": "third"}],
    },
]


# 4. Plusieurs événements au même timestamp
# L'ordre d'origine doit être conservé.
assert build_timeline([
    {"timestamp": 5, "service": "api", "message": "A"},
    {"timestamp": 5, "service": "db", "message": "B"},
    {"timestamp": 5, "service": "worker", "message": "C"},
]) == [
    {
        "timestamp": 5,
        "events": [
            {"service": "api", "message": "A"},
            {"service": "db", "message": "B"},
            {"service": "worker", "message": "C"},
        ],
    }
]


# 5. Cas plus conséquent :
# timestamps mélangés + plusieurs groupes
events = [
    {"timestamp": 12, "service": "api", "message": "timeout"},
    {"timestamp": 5, "service": "db", "message": "connection lost"},
    {"timestamp": 12, "service": "worker", "message": "job failed"},
    {"timestamp": 8, "service": "api", "message": "bad request"},
    {"timestamp": 5, "service": "api", "message": "unauthorized"},
    {"timestamp": 8, "service": "db", "message": "slow query"},
    {"timestamp": 12, "service": "scheduler", "message": "late job"},
    {"timestamp": 2, "service": "worker", "message": "started"},
]

assert build_timeline(events) == [
    {
        "timestamp": 2,
        "events": [
            {"service": "worker", "message": "started"},
        ],
    },
    {
        "timestamp": 5,
        "events": [
            {"service": "db", "message": "connection lost"},
            {"service": "api", "message": "unauthorized"},
        ],
    },
    {
        "timestamp": 8,
        "events": [
            {"service": "api", "message": "bad request"},
            {"service": "db", "message": "slow query"},
        ],
    },
    {
        "timestamp": 12,
        "events": [
            {"service": "api", "message": "timeout"},
            {"service": "worker", "message": "job failed"},
            {"service": "scheduler", "message": "late job"},
        ],
    },
]


# 6. Timestamps négatifs et zéro
assert build_timeline([
    {"timestamp": 0, "service": "a", "message": "zero"},
    {"timestamp": -10, "service": "b", "message": "negative"},
    {"timestamp": 3, "service": "c", "message": "positive"},
    {"timestamp": -10, "service": "d", "message": "negative again"},
]) == [
    {
        "timestamp": -10,
        "events": [
            {"service": "b", "message": "negative"},
            {"service": "d", "message": "negative again"},
        ],
    },
    {
        "timestamp": 0,
        "events": [
            {"service": "a", "message": "zero"},
        ],
    },
    {
        "timestamp": 3,
        "events": [
            {"service": "c", "message": "positive"},
        ],
    },
]


# 7. Deux événements strictement identiques :
# les deux doivent être conservés.
assert build_timeline([
    {"timestamp": 1, "service": "api", "message": "error"},
    {"timestamp": 1, "service": "api", "message": "error"},
]) == [
    {
        "timestamp": 1,
        "events": [
            {"service": "api", "message": "error"},
            {"service": "api", "message": "error"},
        ],
    }
]


# 8. Champ timestamp manquant
try:
    build_timeline([
        {"service": "api", "message": "error"}
    ])
    assert False, "ValueError attendu : timestamp manquant"
except ValueError:
    pass


# 9. Champ service manquant
try:
    build_timeline([
        {"timestamp": 1, "message": "error"}
    ])
    assert False, "ValueError attendu : service manquant"
except ValueError:
    pass


# 10. Champ message manquant
try:
    build_timeline([
        {"timestamp": 1, "service": "api"}
    ])
    assert False, "ValueError attendu : message manquant"
except ValueError:
    pass


# 11. Un événement valide puis un événement invalide
try:
    build_timeline([
        {"timestamp": 1, "service": "api", "message": "ok"},
        {"timestamp": 2, "service": "db"},
        {"timestamp": 3, "service": "worker", "message": "ok"},
    ])
    assert False, "ValueError attendu"
except ValueError:
    pass


# 12. L'entrée ne doit pas être modifiée
events = [
    {"timestamp": 3, "service": "api", "message": "A"},
    {"timestamp": 1, "service": "db", "message": "B"},
    {"timestamp": 3, "service": "worker", "message": "C"},
]

original = [event.copy() for event in events]

build_timeline(events)

assert events == original


print("Tous les tests sont passés.")

Tous les tests sont passés.


In [ ]:
def build_timeline(events: list[dict]) -> list[dict]:
    required_keys = {"timestamp", "service", "message"}
    grouped_events = {}

    for event in events:
        if not required_keys.issubset(event):
            raise ValueError(
                f"Event {event} is missing required keys"
            )

        timestamp = event["timestamp"]

        if timestamp not in grouped_events:
            grouped_events[timestamp] = []

        grouped_events[timestamp].append({
            "service": event["service"],
            "message": event["message"],
        })

    return [
        {
            "timestamp": timestamp,
            "events": grouped_events[timestamp],
        }
        for timestamp in sorted(grouped_events)
    ]